# Fake News Detection

This notebook provides a reproducible **text-classification baseline** for the two CSV files documented in `README.md`. It deliberately fits preprocessing and the model on the training partition only, uses stratified train/validation/test splits, and reports metrics without claiming results when the data is unavailable.

The original notebook trained Word2Vec and an LSTM on all examples before splitting. That made the reported test estimate optimistic and required heavyweight, version-sensitive dependencies. This repaired version uses TF-IDF plus logistic regression as a transparent baseline.


In [ ]:
from pathlib import Path
import os
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score, precision_score, recall_score,
)
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
DATA_ROOT = Path(os.environ.get("FAKE_NEWS_DATA_DIR", "data")).expanduser()
TRUE_FILE = DATA_ROOT / "DataSet_Misinfo_TRUE.csv"
FAKE_FILE = DATA_ROOT / "DataSet_Misinfo_FAKE.csv"


## Load and validate the data

The dataset is intentionally not versioned. Set `FAKE_NEWS_DATA_DIR` to a directory containing `DataSet_Misinfo_TRUE.csv` and `DataSet_Misinfo_FAKE.csv`. The loader fails early with the missing paths and validates the expected `text` column instead of silently training on a different schema.


In [ ]:
def _read_labeled_csv(path: Path, label: int) -> pd.DataFrame:
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing dataset file: {path}. Set FAKE_NEWS_DATA_DIR to the directory "
            "containing both required CSV files."
        )
    frame = pd.read_csv(path)
    frame = frame.drop(columns=[column for column in frame.columns if column.lower().startswith("unnamed:")], errors="ignore")
    if "text" not in frame.columns:
        raise ValueError(f"{path} must contain a 'text' column; found {list(frame.columns)}")
    frame = frame[["text"]].copy()
    frame["label"] = label
    return frame


def load_dataset() -> pd.DataFrame:
    combined = pd.concat(
        [_read_labeled_csv(TRUE_FILE, 1), _read_labeled_csv(FAKE_FILE, 0)],
        ignore_index=True,
    )
    combined["text"] = combined["text"].astype("string").str.strip()
    combined = combined.dropna(subset=["text"])
    combined = combined[combined["text"].str.len() > 0].copy()
    conflicting = (
        combined.groupby("text", dropna=False)["label"].nunique() > 1
    )
    if conflicting.any():
        raise ValueError(
            f"Found {int(conflicting.sum())} text values with conflicting labels; "
            "resolve them before training to avoid ambiguous examples."
        )
    # Exact duplicates can otherwise land in both partitions and leak the label.
    combined = combined.drop_duplicates(subset=["text"]).reset_index(drop=True)
    if combined["label"].nunique() != 2:
        raise ValueError("Both TRUE and FAKE classes must contain at least one usable row.")
    return combined


df = load_dataset()
print(f"Loaded {len(df):,} unique non-empty articles from {DATA_ROOT}")
print(df["label"].value_counts().rename(index={0: "fake", 1: "true"}).sort_index())


In [ ]:
# Inspect class balance without producing or storing a result.
sns.countplot(data=df, x="label", hue="label", legend=False, palette="Set2")
plt.xticks([0, 1], ["Fake", "True"])
plt.xlabel("Label")
plt.ylabel("Articles")
plt.title("Class distribution")
plt.show()


## Split before fitting any learned preprocessing

The test set is held out once. A validation set is carved out of the remaining training data. `stratify` preserves class proportions where the dataset is large enough; scikit-learn will raise a clear error for an undersized dataset rather than silently changing the protocol.


In [ ]:
text_train_val, text_test, y_train_val, y_test = train_test_split(
    df["text"],
    df["label"],
    test_size=0.20,
    random_state=SEED,
    stratify=df["label"],
)
text_train, text_val, y_train, y_val = train_test_split(
    text_train_val,
    y_train_val,
    test_size=0.25,
    random_state=SEED,
    stratify=y_train_val,
)
print(f"train={len(text_train):,}, validation={len(text_val):,}, test={len(text_test):,}")


In [ ]:
# Fit TF-IDF only on training text. Validation and test are transform-only.
vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.98,
    sublinear_tf=True,
)
X_train = vectorizer.fit_transform(text_train)
X_val = vectorizer.transform(text_val)
X_test = vectorizer.transform(text_test)
print(f"TF-IDF features learned from train only: {X_train.shape[1]:,}")


In [ ]:
model = LogisticRegression(
    max_iter=1000,
    random_state=SEED,
    solver="liblinear",
)
model.fit(X_train, y_train)


In [ ]:
def evaluate(split_name, features, labels):
    predictions = model.predict(features)
    metrics = {
        "accuracy": accuracy_score(labels, predictions),
        "balanced_accuracy": balanced_accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions, zero_division=0),
        "recall": recall_score(labels, predictions, zero_division=0),
        "f1": f1_score(labels, predictions, zero_division=0),
    }
    print(f"\n{split_name} metrics")
    print(pd.Series(metrics).to_string(float_format=lambda value: f"{value:.4f}"))
    print(classification_report(labels, predictions, target_names=["fake", "true"], zero_division=0))
    return predictions, metrics


y_val_pred, validation_metrics = evaluate("Validation", X_val, y_val)
y_test_pred, test_metrics = evaluate("Test (held out)", X_test, y_test)


In [ ]:
confusion = confusion_matrix(y_test, y_test_pred, labels=[0, 1])
sns.heatmap(
    confusion,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["fake", "true"],
    yticklabels=["fake", "true"],
)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Held-out test confusion matrix")
plt.show()


## Interpretation

The printed values are computed only when the local dataset is present and this notebook is executed from top to bottom. They are not stored in version control. For a serious study, add a frozen data manifest, repeated or time-based validation, threshold analysis, and comparison against a leak-free neural model before drawing conclusions.
